# AB test retention analysis

In [ ]:
import os
import sys
from pathlib import Path

# Get project root (notebooks/ -> use_case/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# IMPORTANT: Update this path to point to your use_case folder
# For example: project_root = Path(r"C:\path\to\your\use_case")
sys.path.append(str(project_root))
os.chdir(project_root)
os.getcwd()


In [ ]:
from scripts.data_cleaner import remove_rows_with_missing_user_id
from scripts.visualizer import display_distributions, time_series_by_group
from scripts.formatter import abtest_summary_table, export_to_pdf
import pandas as pd
import os


In [ ]:
# Load A/B test data
data_path = project_root / "ressource"
ab_test_data = pd.read_csv(data_path / "ab_test_br.csv", delimiter=';')
print(f"A/B test data loaded. Shape: {ab_test_data.shape}")
ab_test_data.head()


In [ ]:
# Clean A/B test data
ab_test_data = remove_rows_with_missing_user_id(ab_test_data.copy(), user_id_column='keychain_udid')

In [ ]:
# Display distributions of the three variables
distribution_figs = display_distributions(ab_test_data, ['ads_revenue', 'meaningful_reactions', 'sessions'])


In [ ]:
# Time series graphs by group (test/control)
time_series_figs = time_series_by_group(ab_test_data, ['ads_revenue', 'meaningful_reactions', 'sessions'])


In [ ]:
# Create table with means, p-values and significance by group (test/control)
summary_table = abtest_summary_table(ab_test_data, ['ads_revenue', 'meaningful_reactions', 'sessions'])
summary_table


## Global Conclusions

Visually, none of the three variables follow a normal distribution — some are even discrete rather than continuous. When observing the evolution of the gap between test and control groups over time, clear and stable uplifts or downlifts appear, suggesting robust differences.

Before running the statistical tests, we removed the outliers (top 1%) to avoid the influence of extreme users and focus on the main population. Statistical validation was then performed using both t-tests and a non-parametric permutation test. Both approaches led to consistent results.

For Session, no significant effect is observed after removing the outliers. When including them, the metric becomes significant which may indicate a real effect concentrated among a few heavy users. However, since this pattern does not generalize to 99% of the population, Session is considered not significant overall.


> Meaningful Reactions → significant uplift

> Ad Revenue → significant downlift

> Session → not significant, no uplift


In [ ]:
# Export all graphs to PDF
all_figs = distribution_figs + time_series_figs
export_to_pdf(all_figs, filename='ab_test_analysis.pdf')
